In [1]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize


def canonicalize_and_neutralize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")

    # Canonicalize
    canonical = Chem.MolToSmiles(mol, canonical=True)

    # Neutralize
    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(Chem.MolFromSmiles(canonical))

    return Chem.MolToSmiles(mol, canonical=True), mol


def conjugation_length(mol):
    """Longest connected conjugated atom path."""
    graph = {}

    for bond in mol.GetBonds():
        if bond.GetIsConjugated():
            a1 = bond.GetBeginAtomIdx()
            a2 = bond.GetEndAtomIdx()
            graph.setdefault(a1, []).append(a2)
            graph.setdefault(a2, []).append(a1)

    if not graph:
        return 0

    def dfs(node, visited):
        longest = len(visited)
        for nbr in graph.get(node, []):
            if nbr not in visited:
                longest = max(longest, dfs(nbr, visited | {nbr}))
        return longest

    return max(dfs(node, {node}) for node in graph)


def aromatic_surface_fraction(mol):
    heavy = mol.GetNumHeavyAtoms()
    if heavy == 0:
        return 0.0

    aromatic = sum(atom.GetIsAromatic() for atom in mol.GetAtoms())
    return aromatic / heavy


def calculate_properties(smiles):
    canonical_smiles, mol = canonicalize_and_neutralize(smiles)

    return {
        "Canonical SMILES": canonical_smiles,
        "Molecular Weight": round(Descriptors.MolWt(mol), 3),
        "Conjugation Length": conjugation_length(mol),
        "Aromatic Surface Fraction": round(aromatic_surface_fraction(mol), 3),
        "Fsp3": round(rdMolDescriptors.CalcFractionCSP3(mol), 3),
    }


def format_property_line(props):
    return (
        f"MW = {props['Molecular Weight']} | "
        f"ASF = {props['Aromatic Surface Fraction']} | "
        f"CL = {props['Conjugation Length']} | "
        f"Fsp3 = {props['Fsp3']}"
    )


def print_properties(smiles_values):
    for idx, smiles in enumerate(smiles_values, start=1):
        try:
            props = calculate_properties(smiles)
        except ValueError as exc:
            print(f"{idx}. ERROR: {exc} | SMILES = {smiles}")
            continue

        label = "input" if idx % 2 else "output"
        print(f"{idx}. {label}: {format_property_line(props)}")

In [2]:
smiles_list = [
    "CN(C)c1ccc2c(c1)CC1=CC(=[N+](C)C)C(=N2)C=C1", # input
    "CN(C)c1ccc2c(c1)CC1=CC(=C(C)C)[C@H](C2=O)C=N1", # output
    "COc1cc(N)c2c(c1N)C(=O)c1ccccc1C2=O", # input
    "COc1cc(N)c2c(c1)NC(=O)c1ccccc1C2=O", # output
    "Cc1cc(C=Cc2ccc(N(C)C)cc2)cc(C)[o+]1", # input
    "Cc1cc(C=Cc2ccc(N(C)C)cc2)cc(C)n1", # output
    "O=[N+]([O-])c1ccc(Cc2ccncc2)cc1", # input
    "O=S([O-])c1ccc(Cc2ccncc2)cc1"] # output

In [3]:
print_properties(smiles_list)

1. input: MW = 266.368 | ASF = 0.3 | CL = 14 | Fsp3 = 0.294
2. output: MW = 280.371 | ASF = 0.286 | CL = 8 | Fsp3 = 0.333
3. input: MW = 268.272 | ASF = 0.6 | CL = 16 | Fsp3 = 0.067
4. output: MW = 268.272 | ASF = 0.75 | CL = 16 | Fsp3 = 0.067
5. input: MW = 254.353 | ASF = 0.632 | CL = 14 | Fsp3 = 0.235
6. output: MW = 252.361 | ASF = 0.632 | CL = 14 | Fsp3 = 0.235
7. input: MW = 214.224 | ASF = 0.75 | CL = 8 | Fsp3 = 0.083
8. output: MW = 233.292 | ASF = 0.75 | CL = 6 | Fsp3 = 0.083


[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Running Uncharger
[15:03:35] Removed negative charge.
